# EOA Framework — 02. Live Evaluation with Google Gemini API

This notebook executes the **EOA (Explanation-Oriented Agents)** architecture live against the **Google Gemini API**.

**Automatic `.env` Discovery:**
`eoa.default_layer()` and `eoa.ExplanationLayer()` automatically load your `.env` file from the project root (`GEMINI_API_KEY` and optional `GEMINI_MODEL`).

## 1. Setup & Auto-Load `.env` Credentials

In [1]:
import os
import sys
from pathlib import Path

# Ensure project root/src is in Python path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

# Load .env file from project root into os.environ if present
env_file = ROOT_DIR / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

import eoa
from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
    default_layer,
)

# Instantiate zero-config default layer (reads GEMINI_API_KEY from environment)
layer = ExplanationLayer(provider="gemini", model="gemini-3.5-flash")
print(f"EOA v{eoa.__version__} initialized with live model: {layer.llm.model}")

EOA v0.1.0 initialized with live model: gemini-3.5-flash


## 2. Live Scenario 1: Plug & Play Explanation & LLM-as-a-Judge Audit

In [2]:
incident_402 = {
    "prediction": {
        "model_name": "XGBoost_CoolerCondition_v2",
        "target_variable": "Condição do Resfriador Hidráulico",
        "predicted_value": "Eficiência Reduzida (Nível Crítico)",
        "confidence_or_probability": 0.86,
        "status_label": "Operação Degradada / Alerta Crítico",
    },
    "xai": {
        "method": "SHAP",
        "feature_attributions": {
            "PS1_Pressao_Entrada": 0.54,
            "EPS1_Potencia_Motor": 0.32,
            "TS2_Temp_Trocador": 0.14,
        },
    },
    "context": {
        "variables": {
            "PS1_Pressao_Entrada": "Bar, faixa normal de operação [2.0, 4.0] bar",
            "EPS1_Potencia_Motor": "kW, faixa normal de operação [1.5, 2.8] kW",
        },
        "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
        "documents": {
            "POP_Resfriamento_04": "Seção 4: Se PS1 exceder 4.0 bar acompanhado por EPS1 > 3.0 kW, há provável cavitação no filtro de sucção. Reduzir carga hidráulica imediatamente."
        },
    },
}

response = layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Explique por que este alerta disparou e qual ação imediata tomar em campo.",
)

print("=== NARRATIVA REAL GERADA PELO GEMINI ===\n")
print(response.narrative)
print("\n" + "=" * 70)
print(f"Nota da Auditoria (ValidatorAgent): {response.validation_score}/10.0")
print(f"Aprovado no Quality Gate: {response.validation_passed} | Tentativas: {response.attempts_used}")
print("=" * 70)

=== NARRATIVA REAL GERADA PELO GEMINI ===

O sistema de suporte à decisão detectou um estado de **Operação Degradada / Alerta Crítico**, indicando a condição de **Eficiência Reduzida (Nível Crítico)** com uma probabilidade de risco de **86,00%** através do modelo `XGBoost_CoolerCondition_v2`. Esse diagnóstico aponta para um comprometimento severo no desempenho de resfriamento do sistema.

A conclusão do modelo é impulsionada predominantemente pela **`PS1_Pressao_Entrada`** (maior impacto SHAP: +0,5400) e pela **`EPS1_Potencia_Motor`** (+0,3200), respaldadas pela elevação na **`TS2_Temp_Trocador`** (+0,1400). No contexto do procedimento **POP_Resfriamento_04**, o comportamento elevado e simultâneo da pressão de entrada e da potência do motor é o padrão característico de **provável cavitação no filtro de sucção**, condição em que o fluxo restrito degrada diretamente a eficiência operacional do trocador.

Como **ação imediata em campo** estipulada pelo POP_Resfriamento_04, a orientação é 

## 3. Inspecting the Execution Trace (Latency & Agent Steps)

In [3]:
for step in response.execution_trace:
    print(f"[{step.agent_name:<16}] {step.execution_time_ms:>7.1f} ms | status={step.status:<10} | {step.summary}")

[context_manager ] 34386.7 ms | status=success    | Domain context and manual rules structured successfully.
[artifacts       ] 32998.6 ms | status=success    | Prediction and XAI feature attributions analyzed.
[explainer       ]  8960.4 ms | status=success    | Synthesized narrative explanation for user query.
[validator       ] 29538.6 ms | status=success    | Validated: Approved (Score 10.0/7.0).


## 4. Live Scenario 2: Multi-Turn Conversation with SQLite Persistence

In [4]:
store = SQLiteSessionStore("live_gemini_sessions.db")
chat_layer = default_layer(session_store=store)
session_id = "sessao_real_turbina_01"

# Turn 1: Initial diagnosis
res1 = chat_layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Resuma o problema detectado no resfriador.",
    session_id=session_id,
)
print("[TURNO 1]:\n", res1.narrative)

# Turn 2: Operator follow-up question (inherits session artifacts automatically!)
res2 = chat_layer.explain(
    user_query="Se não pudermos parar a bomba agora, qual o risco para o motor elétrico (EPS1)?",
    session_id=session_id,
)
print("\n[TURNO 2 - FOLLOW-UP]:\n", res2.narrative)
print(f"\nScore Turno 2: {res2.validation_score}/10.0 | Mensagens no banco: {len(store.get_history(session_id))}")

[TURNO 1]:
 O sistema de suporte à decisão diagnosticou o resfriador hidráulico em condição de **Eficiência Reduzida (Nível Crítico)**, com 86,00% de confiança pelo modelo *XGBoost_CoolerCondition_v2*, indicando uma operação degradada. 

Esta conclusão é impulsionada positivamente pela pressão de entrada (`PS1_Pressao_Entrada`, com peso de atribuição de +0,5400), pela potência do motor (`EPS1_Potencia_Motor`, com peso de +0,3200) e pela temperatura do trocador (`TS2_Temp_Trocador`, com peso de +0,1400). Segundo o contexto operacional (*POP_Resfriamento_04*), valores elevados nessas variáveis — especificamente quando a pressão excede o limite máximo de 4,0 bar acompanhada de uma potência superior a 3,0 kW — apontam para uma alta probabilidade de cavitação no filtro de sucção.

Como próximo passo, o operador deve verificar imediatamente os valores atuais de `PS1` e `EPS1` nos painéis de controle. Caso os limites normativos estejam comprometidos simultaneamente, é necessário proceder à re

In [5]:
# Turn 3: Follow-up query executed asynchronously (await aexplain)
res3 = await chat_layer.aexplain(
    user_query="Me explica as variáveis que estamos tratando e o que elas representam, você tem alguma tool associada?",
    session_id=session_id,
)

print(f"\n[TURNO 3 - FOLLOW-UP (Async)] (Score: {res3.validation_score}/10.0):\n", res3.narrative)



[TURNO 3 - FOLLOW-UP (Async)] (Score: 10.0/10.0):
 O sistema de suporte à decisão opera exclusivamente com análises de modelos preditivos e diretrizes de domínio, **não possuindo ferramentas automatizadas ou de controle remoto associadas** — todas as intervenções físicas dependem estritamente da ação do operador humano.

As variáveis em tratamento e seus respectivos significados no contexto atual são:
1. **`PS1_Pressao_Entrada`**: Representa a pressão de entrada para o sistema PS1 (medida em Bar), cuja faixa normal de operação está entre 2,0 e 4,0 bar (com limite máximo de 4,0 bar). É a variável de maior impacto positivo (+0,5400) no diagnóstico crítico.
2. **`EPS1_Potencia_Motor`**: Representa a potência do motor elétrico para o EPS1 (medida em kW), com faixa operacional normal entre 1,5 e 2,8 kW e limite máximo de 3,0 kW. Possui peso de atribuição de +0,3200.
3. **`TS2_Temp_Trocador`**: Representa a temperatura do trocador de calor (com impacto positivo de +0,1400), complementando a

## 5. Live Scenario 3: Prompt Override & Objective Follow-Up Responses

In this scenario, we use `chat_layer.override_agent_prompt` to customize the `ExplainerAgent` prompt for concise and direct answers without repeating the entire incident diagnosis.

In [6]:
# Sobrescreve o prompt do Explainer para respostas concisas no chat
chat_layer.override_agent_prompt(
    name="explainer",
    prompt="""
Responda de forma extremamente objetiva e técnica à dúvida do operador, sem rodeios e sem encher linguiça.

Dúvida do operador:
{user_query}
""",
)

# Inspeciona a camada (mostra "Inline (str)" para o Explainer)
chat_layer.inspect()

# Turno 4: Pergunta conceitual de follow-up
res4 = chat_layer.explain(
    user_query="O que é um motor de pressão?",
    session_id=session_id,
)

print(f"\n[TURNO 4 - Pós reescrita no prompt] (Score: {res4.validation_score}/10.0):\n", res4.narrative)


EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Inline (str)
validator          Quality Gate Auditor     0        Built-in


ValidatorAgent rejected draft on attempt 1/3 (score=3.00 < threshold=7.00). Critique: Ajuste a explicação para contextualizar a pergunta do usuário dentro do domínio do sistema hidráulico monitorado (utilizando as variáveis EPS1_Potencia_Motor e PS1_Pressao_Entrada). Explique o conceito relacionando-o com os dados de operação e o alerta crítico do resfriador hidráulico, em vez de fornecer uma definição genérica de motor pneumático.



[TURNO 4 - Pós reescrita no prompt] (Score: 10.0/10.0):
 Um **motor de pressão** (comumente chamado de motor pneumático) é um atuador que converte energia potencial de um fluido sob pressão (ar comprimido) em **energia mecânica rotativa**. 

Seu funcionamento baseia-se na expansão do gás no interior de câmaras seladas acopladas a um rotor, gerando torque e rotação no eixo de saída.


## 6. Live Scenario 4: Specialized Agent with Custom Python Tool Execution

Here we register a `SpecializedAgent` (`math`) with a Python tool function (`soma_fake`), passing query parameters via `query_params={"a": 2, "b": 3}`.

The EOA framework automatically resolves the signature, executes the tool, and injects the output `{tool_outputs}` into the specialist's prompt.

In [10]:
# 1. Restaura o prompt do Explainer para sintetizar pareceres dos especialistas
chat_layer.reset_agent_prompt("explainer")

# 2. Função customizada Python com parâmetros tipados
def soma_fake(a: int, b: int) -> int:
    """Multiplica os fatores simulando uma soma fake customizada."""
    return a * b


# 3. Cria o especialista com o placeholder {tool_outputs}
matematico = SpecializedAgent(
    name="math",
    role="Math Fake",
    prompt="""
Você é um especialista em matemática fake.
A ferramenta executou e calculou o seguinte resultado numérico: {tool_outputs}
Explique o resultado e o conceito de cálculo fake para a dúvida: {user_query}
""",
)

# 4. Registra a ferramenta no especialista e o especialista na camada
matematico.register_tool(soma_fake)
chat_layer.register_agent(matematico)

# Inspeciona a camada (mostra math ativo com 1 Tool e explainer Built-in)
chat_layer.inspect()

# 5. Executa passando os parâmetros exigidos pela ferramenta em query_params:
res5 = chat_layer.explain(
    user_query="Me explica o que é um cálculo fake e apresente o resultado da soma fake para 2 e 3.",
    query_params={"a": 2, "b": 3},  # Parâmetros entregues automaticamente para soma_fake(a=2, b=3) = 6
    session_id=session_id,
)

print("\n[TURNO 5 - Especialista + Tool (soma_fake 2 * 3 = 6)]:\n", res5.narrative)


EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in
math               Math Fake                1        Inline (str)

[TURNO 5 - Especialista + Tool (soma_fake 2 * 3 = 6)]:
 Embora o especialista da nossa base de conhecimento apresente o "cálculo fake" como uma brincadeira matemática baseada em vieses emocionais e conveniência narrativa para justificar resultados como $2 + 3 = 6$, cumpre ressaltar que este sistema de suporte à decisão opera estritamente com base em análises técnicas, modelos preditivos e diretrizes operacionais reais. 

Como sistema analítico rigoroso, não executamos operações a

In [12]:
chat_layer.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in
math               Math Fake                1        Inline (str)


'==============================================================================\nEOA Framework - Active Explanation Layer Configuration\n==============================================================================\nAgent              Role                     Tools    Prompt Source\n------------------------------------------------------------------------------\ncontext_manager    Context Manager          0        Built-in\nartifacts          Artifacts Analyst        0        Built-in\nexplainer          Explainer                0        Built-in\nvalidator          Quality Gate Auditor     0        Built-in\nmath               Math Fake                1        Inline (str)\n=============================================================================='